# 2.5 Optimal Trading Strategy

This notebook implements Section 2.5 of the project by reusing the outputs and helper functions from Sections 2.1--2.4.

The objective is to build trading strategies from the synthetic alpha signals, including a synthetic overnight alpha, and then backtest these strategies with the existing backtest engine.

The implementation is designed to be consistent with the lecture notes. In the Obizhaeva--Wang model, impact satisfies

$$
dI_t = -\beta I_t dt + \lambda dQ_t.
$$

For a deterministic alpha signal, the lecture notes derive the optimal target impact state

$$
I_t^* = \frac{1}{2}\left(\alpha_t - \beta^{-1}\alpha_t'\right).
$$

Trades are then recovered from the impact dynamics:

$$
dQ_t^* = \frac{1}{\lambda}\left(\beta I_t^* dt + dI_t^*\right).
$$

In the fitted implementation from Sections 2.2--2.3, the impact state is normalized as

$$
I_{n+1}=e^{-\beta\Delta t} I_n + \lambda_{\text{hat}}\,\sigma\,\frac{\Delta Q_n}{ADV}.
$$

Therefore, after computing a target impact state from alpha, we recover trades using the discrete fitted version:

$$
\Delta Q_n
=
\frac{I_n^* - e^{-\beta\Delta t}I_{n-1}^*}
{\lambda_{\text{hat}}\sigma / ADV}.
$$

## 1. Imports and paths

This notebook assumes that the helper `.py` files and the saved results from Sections 2.1--2.4 are available in the same folder as this notebook.

In [13]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
import src.data_prep as data_prep
import src.backtest_engine as backtest_engine

DATA_DIR = r"data"

sys.path.append(str(DATA_DIR))

from src.data_prep import *
from src.backtest_engine import *
from src.optimal_trading_strategies import *

## 2. Load outputs from previous sections

We reuse:

- Section 2.1: prepared train/test price panels and test traded-volume panels;
- Section 2.2: fitted impact model parameters;
- Section 2.3: backtest engine;
- Section 2.4: synthetic intraday alpha panel.

In [9]:

train_px_df = load_panel_csv(os.path.join(DATA_DIR, "train_px_201901_20.csv"))
test_px_df = load_panel_csv(os.path.join(DATA_DIR, "test_px_201902_20.csv"))
test_traded_volume_df = load_panel_csv(os.path.join(DATA_DIR, "test_traded_volume_201902_20.csv"))

scaling_df = load_stock_level_csv(os.path.join(DATA_DIR, "scaling_201901_20.csv"))

best_df = pd.read_csv(os.path.join(DATA_DIR, "impact_model_best_by_is_201901_train_201902_test.csv"))
stock_results_df = pd.read_csv(os.path.join(DATA_DIR, "impact_model_stock_results_201901_train_201902_test.csv"))

synthetic_alpha_df = load_panel_csv(os.path.join(DATA_DIR, "synthetic_alpha_201902_20.csv"))
synthetic_alpha_diagnostics_df = load_stock_level_csv(os.path.join(DATA_DIR, "synthetic_alpha_diagnostics_201902_20.csv"))

# Keep only common intraday columns across all objects.
test_px_df, test_traded_volume_df, synthetic_alpha_df = align_intraday_columns(
    test_px_df,
    test_traded_volume_df,
    synthetic_alpha_df,
)

print("test_px_df:", test_px_df.shape)
print("test_traded_volume_df:", test_traded_volume_df.shape)
print("synthetic_alpha_df:", synthetic_alpha_df.shape)
print("scaling_df:", scaling_df.shape)
print("best_df:", best_df.shape)
print("stock_results_df:", stock_results_df.shape)

test_px_df: (380, 2341)
test_traded_volume_df: (380, 2341)
synthetic_alpha_df: (380, 2341)
scaling_df: (20, 2)
best_df: (3, 9)
stock_results_df: (420, 25)


## 3. Reconstruct fitted model parameter tables

Section 2.2 selected one best half-life for each model type. Section 2.3 uses `get_best_model_fit_df` to extract stock-level fitted parameters for that selected half-life.

In [10]:
model_types = ["ow", "afs", "reduced_form"]

fit_dfs = {}
for model_type in model_types:
    fit_dfs[model_type] = get_best_model_fit_df(
        stock_results_df=stock_results_df,
        best_df=best_df,
        model_type=model_type,
    )

for model_type, fit_df in fit_dfs.items():
    print(model_type, fit_df.shape)
    display(fit_df[["lambda_hat", "half_life_seconds", "is_r2", "oos_r2"]].head())

ow (20, 24)


,lambda_hat,half_life_seconds,is_r2,oos_r2
stock,,,,
AAL,262.866026,1800,0.143934,0.199819
AAPL,407.392206,1800,0.231705,0.252164
ABBV,382.822089,1800,0.130716,0.131900
ABT,440.867448,1800,0.171972,0.115936
ADBE,339.189338,1800,0.123732,0.137870


afs (20, 24)


,lambda_hat,half_life_seconds,is_r2,oos_r2
stock,,,,
AAL,0.770608,300,0.151269,0.185629
AAPL,0.828313,300,0.234038,0.227081
ABBV,0.540337,300,0.102414,0.115905
ABT,0.550386,300,0.133748,0.099200
ADBE,0.573366,300,0.130973,0.123505


reduced_form (20, 24)


,lambda_hat,half_life_seconds,is_r2,oos_r2
stock,,,,
AAL,73.752339,600,0.181003,0.244088
AAPL,106.653594,600,0.281354,0.309429
ABBV,89.711831,600,0.151770,0.123431
ABT,99.691282,600,0.186580,0.098858
ADBE,87.620862,600,0.156239,0.175975


## 4. Synthetic overnight alpha

The lecture notes suggest comparing execution slippage to the overnight return

$$
\frac{P_{\text{next open}} - P_{\text{close}}}{P_{\text{close}}}.
$$

For each stock-date in the test set, we define a synthetic overnight alpha as the next open return. This is a look-ahead construction, consistent with the synthetic alpha design in Section 2.4.

The overnight alpha is constant across the intraday grid for a given stock-day:

$$
\alpha^{ON}_{d,t}=A_{ON}\frac{P_{d+1,\text{open}}-P_{d,\text{close}}}{P_{d,\text{close}}}.
$$

In [14]:
overnight_alpha_level = 1.0
overnight_alpha_df, overnight_alpha_series = make_next_open_overnight_alpha_panel(
    test_px_df=test_px_df,
    overnight_alpha_level=overnight_alpha_level,
)

print("overnight_alpha_df:", overnight_alpha_df.shape)
display(overnight_alpha_series.groupby(level="stock").describe().head())

NameError: name 'make_next_open_overnight_alpha_panel' is not defined

## 5. Alpha scenarios

We construct four alpha scenarios:

1. **No alpha**: zero signal. Under the unconstrained OW alpha-trading problem, this produces no trades, so it is used only as a conceptual benchmark.
2. **Overnight alpha only**: uses only the next-open overnight signal.
3. **Intraday alpha only**: uses the synthetic intraday alpha from Section 2.4.
4. **Combined alpha**: uses both overnight and intraday alpha.

The project instruction specifically asks us to simulate strategies with and without intraday alpha. These correspond to scenarios 2 and 4.

For the backtest, we exclude `no_alpha` because it has zero traded notional and therefore undefined bps metrics.

In [ ]:
alpha_scenarios = {
    "no_alpha": synthetic_alpha_df * 0.0,
    "overnight_only": overnight_alpha_df,
    "intraday_only": synthetic_alpha_df,
    "combined": synthetic_alpha_df + overnight_alpha_df,
}

for name, alpha_df in alpha_scenarios.items():
    print(name, alpha_df.shape, "mean abs alpha:", alpha_df.abs().stack().mean())

## 6. OW optimal strategy from target impact

The lecture notes formulate the solution in impact space:

$$
I_t^* = \frac{1}{2}\left(\alpha_t - \beta^{-1}\alpha_t'\right).
$$

Because the previous notebooks fit impact in return units as

$$
I_{n+1}=e^{-\beta\Delta t} I_n + \lambda_{\text{hat}}\sigma\frac{\Delta Q_n}{ADV},
$$

we recover trades from

$$
\Delta Q_n
=
\frac{I_n^* - e^{-\beta\Delta t}I_{n-1}^*}
{\lambda_{\text{hat}}\sigma/ADV}.
$$

The optional normalization below keeps total absolute daily traded volume comparable across alpha scenarios. This is useful for backtest comparison because the analytical OW alpha strategy is unconstrained and otherwise chooses its own trade size from the alpha magnitude.

In [ ]:
def ow_target_impact_from_alpha(alpha, beta, dt_seconds):
    """
    Compute the lecture-note OW target impact state:
        I*_t = 1/2 (alpha_t - beta^{-1} alpha'_t).

    alpha is a pd.Series indexed by intraday time, in return units.
    beta is the impact decay speed in seconds^{-1}.
    """
    alpha = alpha.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    alpha_prime = alpha.diff() / dt_seconds
    if len(alpha_prime) > 1:
        alpha_prime.iloc[0] = alpha_prime.iloc[1]
    else:
        alpha_prime.iloc[0] = 0.0

    I_star = 0.5 * (alpha - alpha_prime / beta)

    # Terminal condition from the deterministic-alpha OW formula.
    I_star.iloc[-1] = alpha.iloc[-1]

    return I_star, alpha_prime


def recover_ow_trades_from_target_impact(
    I_star,
    lambda_hat,
    ADV,
    sigma,
    half_life_seconds,
    dt_seconds=10,
    eps=1e-12,
):
    """
    Recover trades using the discrete fitted OW impact equation:
        I_n = decay * I_{n-1} + lambda_hat * sigma * q_n / ADV.
    """
    beta = np.log(2) / half_life_seconds
    decay = np.exp(-beta * dt_seconds)

    lambda_eff = lambda_hat * sigma / ADV
    if abs(lambda_eff) < eps:
        return pd.Series(0.0, index=I_star.index)

    trades = []
    prev_I = 0.0

    for target_I in I_star.values.astype(float):
        q = (target_I - decay * prev_I) / lambda_eff
        trades.append(q)
        prev_I = target_I

    return pd.Series(trades, index=I_star.index)


def make_ow_optimal_trade_df(
    alpha_df,
    test_px_df,
    scaling_df,
    fit_df,
    dt_seconds=10,
    normalize_abs_volume=True,
    target_participation=0.20,
):
    """
    Build a stock-date x time trade panel from an alpha panel using the
    lecture-note OW target-impact formula and fitted OW parameters.

    If normalize_abs_volume=True, each stock-day is rescaled so that
    total absolute traded volume equals target_participation * ADV.
    This preserves the OW trading shape but makes scenarios comparable.
    """
    trades_df = pd.DataFrame(0.0, index=test_px_df.index, columns=test_px_df.columns)
    target_impact_df = pd.DataFrame(0.0, index=test_px_df.index, columns=test_px_df.columns)
    alpha_prime_df = pd.DataFrame(0.0, index=test_px_df.index, columns=test_px_df.columns)

    for stock, date in test_px_df.index:
        if stock not in fit_df.index or stock not in scaling_df.index:
            continue

        alpha = alpha_df.loc[(stock, date)].reindex(test_px_df.columns).fillna(0.0)

        lambda_hat = float(fit_df.loc[stock, "lambda_hat"])
        half_life_seconds = float(fit_df.loc[stock, "half_life_seconds"])
        beta = np.log(2) / half_life_seconds

        ADV = float(scaling_df.loc[stock, "ADV"])
        sigma = float(scaling_df.loc[stock, "sigma"])

        I_star, alpha_prime = ow_target_impact_from_alpha(
            alpha=alpha,
            beta=beta,
            dt_seconds=dt_seconds,
        )

        trades = recover_ow_trades_from_target_impact(
            I_star=I_star,
            lambda_hat=lambda_hat,
            ADV=ADV,
            sigma=sigma,
            half_life_seconds=half_life_seconds,
            dt_seconds=dt_seconds,
        )

        if normalize_abs_volume:
            target_abs_volume = target_participation * ADV
            current_abs_volume = trades.abs().sum()

            if current_abs_volume > 0:
                trades = trades * target_abs_volume / current_abs_volume

        trades_df.loc[(stock, date)] = trades.values
        target_impact_df.loc[(stock, date)] = I_star.values
        alpha_prime_df.loc[(stock, date)] = alpha_prime.values

    return trades_df, target_impact_df, alpha_prime_df

## 7. Generate OW optimal trades for backtest alpha scenarios

We generate the OW optimal trading shape using the fitted OW parameters from Section 2.2. We normalize total absolute daily traded volume to 20% ADV to keep the backtests comparable with the TWAP benchmarks from Section 2.3.

We backtest `overnight_only`, `intraday_only`, and `combined`. The zero-alpha case is not included because the unconstrained alpha strategy simply does not trade.

In [ ]:
dt_seconds = 10
target_participation = 0.20

ow_fit_df = fit_dfs["ow"]

strategy_trade_dfs = {}
target_impact_dfs = {}
alpha_prime_dfs = {}

backtest_alpha_scenario_names = ["overnight_only", "intraday_only", "combined"]

for scenario_name in backtest_alpha_scenario_names:
    alpha_df = alpha_scenarios[scenario_name]
    trades_df, target_impact_df, alpha_prime_df = make_ow_optimal_trade_df(
        alpha_df=alpha_df,
        test_px_df=test_px_df,
        scaling_df=scaling_df,
        fit_df=ow_fit_df,
        dt_seconds=dt_seconds,
        normalize_abs_volume=True,
        target_participation=target_participation,
    )

    strategy_trade_dfs[scenario_name] = trades_df
    target_impact_dfs[scenario_name] = target_impact_df
    alpha_prime_dfs[scenario_name] = alpha_prime_df

    print(scenario_name)
    print("  total abs traded:", trades_df.abs().sum(axis=1).describe()[["mean", "50%", "min", "max"]].to_dict())
    print("  net traded:", trades_df.sum(axis=1).describe()[["mean", "50%", "min", "max"]].to_dict())

## 8. TWAP benchmark from Section 2.3

We also reuse the round-trip TWAP benchmark from the backtest engine. This is not alpha-based, but it is useful as a sanity-check benchmark.

In [ ]:
twap_round_trip_trades_df = make_round_trip_twap_trade_df(
    test_px_df=test_px_df,
    scaling_df=scaling_df,
    participation_rate=target_participation,
)

strategy_trade_dfs["round_trip_twap"] = twap_round_trip_trades_df

print("round_trip_twap")
print("  total abs traded:", twap_round_trip_trades_df.abs().sum(axis=1).describe()[["mean", "50%", "min", "max"]].to_dict())
print("  net traded:", twap_round_trip_trades_df.sum(axis=1).describe()[["mean", "50%", "min", "max"]].to_dict())

## 9. Backtest strategies using the Section 2.3 engine

The previous section's backtest engine first removes public market impact, then simulates our strategy trades under a fitted impact model.

We first backtest the strategies under the fitted OW model. Then, optionally, we can also run the same trade schedules under the AFS and reduced-form fitted models to check model robustness.

In [ ]:
def run_strategy_backtests_for_model(
    strategy_trade_dfs,
    model_type,
    test_px_df,
    test_traded_volume_df,
    scaling_df,
    fit_dfs,
    dt_seconds=10,
):
    results = []

    for strategy_name, trades_df in strategy_trade_dfs.items():
        bt = run_backtest_from_trade_df(
            test_px_df=test_px_df,
            strategy_trades_df=trades_df,
            test_traded_volume_df=test_traded_volume_df,
            scaling_df=scaling_df,
            fit_df=fit_dfs[model_type],
            model_type=model_type,
            dt_seconds=dt_seconds,
        )

        bt = add_bps_metrics(bt)
        bt["strategy"] = strategy_name
        bt["backtest_model"] = model_type

        results.append(bt)

    return pd.concat(results, ignore_index=True)


ow_strategy_backtests_df = run_strategy_backtests_for_model(
    strategy_trade_dfs=strategy_trade_dfs,
    model_type="ow",
    test_px_df=test_px_df,
    test_traded_volume_df=test_traded_volume_df,
    scaling_df=scaling_df,
    fit_dfs=fit_dfs,
    dt_seconds=dt_seconds,
)

summary_cols = [
    "pnl_bps",
    "impact_cost_bps",
    "max_abs_impact",
    "total_abs_traded",
    "net_traded",
]

ow_summary = (
    ow_strategy_backtests_df
    .groupby("strategy")[summary_cols]
    .agg(["mean", "median", "std"])
)

display(ow_summary)

## 10. Plot strategy comparison

In [ ]:
def plot_strategy_boxplot(backtest_df, metric, title, ylabel):
    strategies = backtest_df["strategy"].unique()
    data = [
        backtest_df.loc[backtest_df["strategy"] == s, metric].dropna()
        for s in strategies
    ]

    plt.figure(figsize=(10, 4))
    plt.boxplot(data, tick_labels=strategies)
    plt.xticks(rotation=30, ha="right")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


plot_strategy_boxplot(
    ow_strategy_backtests_df,
    metric="pnl_bps",
    title="OW backtest: P&L by strategy",
    ylabel="Daily P&L / traded notional, bps",
)

plot_strategy_boxplot(
    ow_strategy_backtests_df,
    metric="impact_cost_bps",
    title="OW backtest: impact cost by strategy",
    ylabel="Impact cost / traded notional, bps",
)

## 11. One-stock-day diagnostic

The plots below show how the alpha signal maps into target impact and trades for one stock-day.

In [ ]:
stock = "AAPL"
# Pick the first available AAPL test date if the chosen stock exists.
if stock in test_px_df.index.get_level_values("stock"):
    date = test_px_df.loc[stock].index[0]
else:
    stock, date = test_px_df.index[0]

scenario_name = "combined"

alpha_one = alpha_scenarios[scenario_name].loc[(stock, date)]
I_star_one = target_impact_dfs[scenario_name].loc[(stock, date)]
trades_one = strategy_trade_dfs[scenario_name].loc[(stock, date)]
position_one = trades_one.cumsum()

x = np.arange(len(alpha_one))
tick_positions = np.linspace(0, len(alpha_one) - 1, 8, dtype=int)
tick_labels = alpha_one.index[tick_positions]

plt.figure(figsize=(12, 4))
plt.plot(x, 100 * alpha_one.values, label="Alpha signal")
plt.plot(x, 100 * I_star_one.values, label="Target impact")
plt.xticks(tick_positions, tick_labels, rotation=45)
plt.ylabel("Return units, %")
plt.title(f"{stock} {date} - alpha and OW target impact ({scenario_name})")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(x, trades_one.values, label="Trade")
plt.plot(x, position_one.values, label="Position")
plt.xticks(tick_positions, tick_labels, rotation=45)
plt.title(f"{stock} {date} - OW trades and inventory ({scenario_name})")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Optional robustness: backtest same trade schedules under all fitted models

The OW formula produces the trade schedules. We can then use the existing backtest engine to evaluate these schedules under each fitted impact model from Section 2.2.

In [ ]:
run_all_model_robustness = False

if run_all_model_robustness:
    all_model_results = []

    for model_type in model_types:
        model_bt = run_strategy_backtests_for_model(
            strategy_trade_dfs=strategy_trade_dfs,
            model_type=model_type,
            test_px_df=test_px_df,
            test_traded_volume_df=test_traded_volume_df,
            scaling_df=scaling_df,
            fit_dfs=fit_dfs,
            dt_seconds=dt_seconds,
        )
        all_model_results.append(model_bt)

    all_model_strategy_backtests_df = pd.concat(all_model_results, ignore_index=True)

    display(
        all_model_strategy_backtests_df
        .groupby(["backtest_model", "strategy"])[["pnl_bps", "impact_cost_bps"]]
        .agg(["mean", "median", "std"])
    )
else:
    print("Set run_all_model_robustness=True to run robustness backtests under AFS and reduced-form models.")

## 13. Save Section 2.5 outputs

In [ ]:
OUTPUT_DIR = DATA_DIR

# Save trade panels.
for strategy_name, trades_df in strategy_trade_dfs.items():
    out = trades_df.reset_index()
    out.to_csv(
        os.path.join(OUTPUT_DIR, f"strategy_trades_{strategy_name}_201902_20.csv"),
        index=False,
    )

# Save main OW backtest comparison.
ow_strategy_backtests_df.to_csv(
    os.path.join(OUTPUT_DIR, "backtest_ow_optimal_alpha_strategies_201902_20.csv"),
    index=False,
)

# Save compact summary.
ow_summary.to_csv(
    os.path.join(OUTPUT_DIR, "backtest_ow_optimal_alpha_strategy_summary_201902_20.csv")
)

print("Saved Section 2.5 outputs.")

## Interpretation

This section extends the baseline backtest in three ways.

First, it uses the synthetic intraday alpha from Section 2.4 as the directional input to the trading strategy.

Second, it adds a synthetic overnight alpha based on the next-open return. This corresponds to a long-horizon directional signal that is constant throughout the intraday trading window.

Third, it converts alpha into trades using the OW optimal strategy from the lecture notes. The strategy is expressed through a target impact state rather than through a direct heuristic schedule such as TWAP or VWAP.

The central economic interpretation is:

- stronger alpha increases the target impact state;
- faster alpha decay increases trading urgency through the alpha derivative term;
- overnight alpha generates a persistent directional bias;
- intraday alpha dynamically changes the trading trajectory during the day;
- the fitted impact model determines how much volume is required to reach the target impact.

The most important comparison for the project is between:

- `overnight_only`: synthetic overnight alpha without intraday alpha;
- `combined`: synthetic overnight alpha with intraday alpha.

This directly answers the instruction to simulate trading strategies with and without the intraday alpha.